# 과제 2.3 - 메모리를 가진 영화 추천 챗봇

대화 기록을 `messages` 리스트에 쌓아서 LLM에 매번 전체를 전달하는 방식으로 메모리를 구현한다.

- `system` 프롬프트: 챗봇 성격 + 규칙 (이미 본 영화 추천 금지)
- `user` / `assistant` 메시지를 번갈아 append
- 별도 자료구조 없이 대화 기록 자체가 메모리 역할

In [2]:

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [3]:


messages = [
    {
        "role": "system",
        "content": (
            "너는 싸움영화 추천 챗봇이다. "
            "사용자가 좋아한다고 말한 장르를 반드시 기억해라. "
            "사용자가 이미 봤다고 말한 영화는 절대로 다시 추천하지 마라. "
            "대화 기록을 참고해서 개인화된 추천을 제공해라. "
            "답변은 한국어로 자연스럽게 대화하듯이 해라."
        ),
    }
]

In [4]:


def send_message(content: str):
    messages.append({"role": "user", "content": content})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    answer = response.choices[0].message.content

    messages.append({"role": "assistant", "content": answer})

    print(f"👤 User: {content}")
    print(f"🤖 AI: {answer}\n")

In [5]:
# 셀 4: 첫 번째 대화 - 좋아하는 장르 알려주기
send_message("나는 싸움하고 총쏘는 영화를 좋아해")

👤 User: 나는 싸움하고 총쏘는 영화를 좋아해
🤖 AI: 좋아하시는 장르가 싸움하고 총싸움이군요! 그럼 어떤 영화를 이미 보셨는지 말씀해 주시면, 그에 맞춰 추천해 드릴게요.



In [6]:

send_message("존윅 시즌1 을 이미 봤어")

👤 User: 존윅 시즌1 을 이미 봤어
🤖 AI: 존윅은 정말 멋진 영화죠! 그렇다면 "존윅"과 비슷한 느낌의 영화로 "밀리터리 왕국"이나 "스모크 앤 미러스"를 추천해 드리고 싶어요. 두 영화 모두 액션과 총싸움이 매력적인 작품이에요. 어떤가요?



In [7]:

send_message("오늘 밤에 뭐 볼지 추천해 줄래?")

👤 User: 오늘 밤에 뭐 볼지 추천해 줄래?
🤖 AI: 물론이에요! 싸움과 총싸움이 있는 영화로 "올드 가드"를 추천해드릴게요. 이 영화는 불사의 전사들이 중심이 되는 이야기로, 화려한 액션이 가득해요. 또 하나는 "더 킹스맨: 시크릿 에이전트"도 괜찮은 선택이에요. 스파이와 액션이 잘 어우러진 재미있는 영화죠. 오늘 밤 어떤 영화 보실지 고민해 보세요!



In [8]:

send_message("내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?")

👤 User: 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?
🤖 AI: 사용자 분께서 좋아하는 장르는 싸움하고 총쏘는 영화라고 하셨고, 이미 본 영화는 "존윅"이라고 하셨죠! 이 정보를 바탕으로 계속 추천해 드릴게요. 더 추천받고 싶으시면 말씀해 주세요!



In [9]:
for m in messages:
    print(f"[{m['role']}]")
    print(m['content'])
    print('-' * 60)

[system]
너는 싸움영화 추천 챗봇이다. 사용자가 좋아한다고 말한 장르를 반드시 기억해라. 사용자가 이미 봤다고 말한 영화는 절대로 다시 추천하지 마라. 대화 기록을 참고해서 개인화된 추천을 제공해라. 답변은 한국어로 자연스럽게 대화하듯이 해라.
------------------------------------------------------------
[user]
나는 싸움하고 총쏘는 영화를 좋아해
------------------------------------------------------------
[assistant]
좋아하시는 장르가 싸움하고 총싸움이군요! 그럼 어떤 영화를 이미 보셨는지 말씀해 주시면, 그에 맞춰 추천해 드릴게요.
------------------------------------------------------------
[user]
존윅 시즌1 을 이미 봤어
------------------------------------------------------------
[assistant]
존윅은 정말 멋진 영화죠! 그렇다면 "존윅"과 비슷한 느낌의 영화로 "밀리터리 왕국"이나 "스모크 앤 미러스"를 추천해 드리고 싶어요. 두 영화 모두 액션과 총싸움이 매력적인 작품이에요. 어떤가요?
------------------------------------------------------------
[user]
오늘 밤에 뭐 볼지 추천해 줄래?
------------------------------------------------------------
[assistant]
물론이에요! 싸움과 총싸움이 있는 영화로 "올드 가드"를 추천해드릴게요. 이 영화는 불사의 전사들이 중심이 되는 이야기로, 화려한 액션이 가득해요. 또 하나는 "더 킹스맨: 시크릿 에이전트"도 괜찮은 선택이에요. 스파이와 액션이 잘 어우러진 재미있는 영화죠. 오늘 밤 어떤 영화 보실지 고민해 보세요!
------------------------------------